In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

test = pd.read_csv('Full_cleaned_test.csv')
train = pd.read_csv('Full_cleaned_train.csv')

In [5]:
# Feature Engineering
for df in [train, test]:
    df['power_output'] = df['voltage'] * df['current']
    df['temp_diff'] = df['module_temperature'] - df['temperature']
    df['irradiance_per_cloud'] = df['irradiance'] / (df['cloud_coverage'] + 1)
    
train['power_output'] = train['voltage'] * train['current']
train['temp_diff'] = train['module_temperature'] - train['temperature']
train['irradiance_per_cloud'] = train['irradiance'] / (train['cloud_coverage'] + 1)

    
test['power_output'] = test['voltage'] * test['current']
test['temp_diff'] = test['module_temperature'] - test['temperature']
test['irradiance_per_cloud'] = test['irradiance'] / (test['cloud_coverage'] + 1)

In [6]:
train.shape

(19369, 20)

In [7]:

# Feature selection
drop_cols = ['id', 'efficiency']
X = train.drop(columns=drop_cols)
y = train['efficiency']
X_test = test.drop(columns=['id'])

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [8]:


# 1. Best params: {'depth': 4, 'iterations': 500, 'learning_rate': 0.03}
model = CatBoostRegressor(iterations=500, learning_rate=0.03, depth=4, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.8f}")

RMSE: 0.0464, Score: 95.36019608


In [9]:

from datetime import datetime
# Get current date and time
now = datetime.now()
timestamp = now.strftime("%H%M%S")
print(f"submission_{timestamp}")

# Final prediction
preds = model.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'efficiency': preds})

submission.to_csv(f'submission_{timestamp}.csv', index=False)

submission_130021


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler

# Scale numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1)  # No activation for regression
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.fit(X_scaled, y, epochs=100, batch_size=64, validation_split=0.2)
